# 🎯 Notebook 3 — Budget vs Actual Analysis

We compare what was actually spent in each category against the monthly budget.
This is the most practically useful analysis for personal finance management.

**Key metric:** Variance = Actual − Budget
- Positive variance = over budget (bad)
- Negative variance = under budget (good)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

tx  = pd.read_csv('../data/cleaned.csv', parse_dates=['date'])
bud = pd.read_csv('../data/budgets.csv')
expenses = tx[tx['type']=='Expense'].copy()
print('Data loaded ✅')

## Step 1 — Build Budget vs Actual Table

In [ ]:
cat_monthly = expenses.groupby(['month','month_name','category'])['amount'].sum().reset_index().rename(columns={'amount':'actual'})
cat_monthly = cat_monthly.merge(bud, on='category', how='left')
cat_monthly['variance']      = cat_monthly['actual'] - cat_monthly['monthly_budget']
cat_monthly['over_budget']   = cat_monthly['variance'] > 0
cat_monthly['pct_of_budget'] = (cat_monthly['actual'] / cat_monthly['monthly_budget'] * 100).round(1)

print('Budget vs Actual (sorted by variance):')
display_cols = ['month_name','category','monthly_budget','actual','variance','pct_of_budget']
cat_monthly.sort_values('variance', ascending=False)[display_cols].head(15)

## Step 2 — Grouped Bar Chart: Budget vs Actual per Category

The red bars show where spending exceeded the budget. The white dashed line at 100% is the budget limit.

In [ ]:
month_order = ['January','February','March']
cat_monthly['month_name'] = pd.Categorical(cat_monthly['month_name'], categories=month_order, ordered=True)

for month in month_order:
    data = cat_monthly[cat_monthly['month_name']==month].copy()
    fig = px.bar(data, x='category', y=['monthly_budget','actual'],
                 barmode='group',
                 title=f'Budget vs Actual — {month}',
                 labels={'value':'Amount (R)','variable':'','category':'Category'},
                 color_discrete_map={'monthly_budget':'#555555','actual':'#ff6b00'},
                 template='plotly_dark')
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()

## Step 3 — Variance Heatmap

A heatmap showing overspend (red) and underspend (green) per category per month at a glance.

In [ ]:
pivot = cat_monthly.pivot(index='category', columns='month_name', values='variance')
pivot = pivot[month_order]

fig = px.imshow(pivot, title='Budget Variance Heatmap (Red = Over, Green = Under)',
                color_continuous_scale='RdYlGn_r',
                labels={'color':'Variance (R)'},
                template='plotly_dark',
                text_auto=True)
fig.update_traces(texttemplate='R %{z:,.0f}')
fig.show()

## Step 4 — Worst Offending Categories

Categories that went over budget most often or by the largest amount.

In [ ]:
over = cat_monthly[cat_monthly['over_budget']].copy()
over_summary = over.groupby('category').agg(
    times_over=('over_budget','count'),
    total_overspend=('variance','sum'),
    avg_overspend=('variance','mean')
).reset_index().sort_values('total_overspend', ascending=False)

print('Categories that went over budget:')
print(over_summary.to_string(index=False))

fig = px.bar(over_summary, x='category', y='total_overspend',
             color='times_over',
             title='Total Overspend by Category (3 Months)',
             labels={'total_overspend':'Total Overspend (R)','times_over':'Months Over Budget'},
             color_continuous_scale='Reds',
             template='plotly_dark')
fig.show()

## Step 5 — 3-Month Budget Compliance Summary

In [ ]:
total_budget   = bud['monthly_budget'].sum() * 3
total_actual   = expenses['amount'].sum()
total_variance = total_actual - total_budget
compliance_pct = (cat_monthly[~cat_monthly['over_budget']].shape[0] /
                  cat_monthly.shape[0] * 100)

print('='*55)
print('3-MONTH BUDGET COMPLIANCE SUMMARY')
print('='*55)
print(f'Total budget (3 months)   : R {total_budget:,.2f}')
print(f'Total actual spend        : R {total_actual:,.2f}')
print(f'Overall variance          : R {total_variance:+,.2f}')
print(f'Budget compliance rate    : {compliance_pct:.1f}%')
print(f'  (% of category-months within budget)')
print('='*55)